In [ ]:
"""GAID manifest CSV generator.

Scans one or more source roots (each laid out as
`<source_root>/<real|fake>/<sub_folder>/<image_files>`) and writes a single CSV.

Per-row schema:
    filepath, filename, label, dataset

`fake/` -> label 0, `real/` -> label 1.

- Multiple source roots supported (add to `SOURCE_ROOTS`).
- Recursively scans every image file under each `real/` and `fake/` split.
- Skips empty folders / non-image files / missing splits.
"""

from pathlib import Path

import pandas as pd

SOURCE_ROOTS = [
    Path("/home/taiaburrahman/dataset_manager_pro/data/processed/v9"),
    Path("/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing"),
]

OUT_DIR = Path("/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv")
OUT_CSV = OUT_DIR / "GAID_Dataset_v11_full_train.csv"

LABEL_MAP = {"fake": 0, "real": 1}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".jfif", ".gif", ".bmp", ".tiff"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

print(f"Source roots : {len(SOURCE_ROOTS)}")
for r in SOURCE_ROOTS:
    print(f"  - {r}  (exists={r.exists()})")
print(f"Output CSV   : {OUT_CSV}")

Source roots : 2
  - /home/taiaburrahman/dataset_manager_pro/data/processed/v9  (exists=True)
  - /home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing  (exists=True)
Output CSV   : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv/GAID_Dataset_v11_full_train.csv


In [2]:
def scan_source(root: Path, dataset_name: str) -> list[dict]:
    """Walk `<root>/<real|fake>/**/*` and collect image rows.

    Empty folders and non-image files are skipped naturally.
    """
    rows: list[dict] = []
    if not root.exists():
        print(f"[warn] missing source root: {root}")
        return rows

    for label_name, label_value in LABEL_MAP.items():
        split_root = root / label_name
        if not split_root.exists():
            print(f"[warn] missing split: {split_root}")
            continue

        before = len(rows)
        for f in split_root.rglob("*"):
            if not f.is_file():
                continue
            if f.suffix.lower() not in IMAGE_EXTS:
                continue
            rows.append(
                {
                    "filepath": str(f),
                    "filename": f.name,
                    "label": label_value,
                    "dataset": dataset_name,
                }
            )
        print(f"  {dataset_name}/{label_name:<4} -> {len(rows) - before:>8,} images")

    return rows


def build_manifest(source_roots: list[Path]) -> pd.DataFrame:
    all_rows: list[dict] = []
    for root in source_roots:
        dataset_name = root.name
        print(f"[scan] {root}  (dataset={dataset_name})")
        all_rows.extend(scan_source(root, dataset_name))
    return pd.DataFrame(
        all_rows, columns=["filepath", "filename", "label", "dataset"]
    )


def save_csv(df: pd.DataFrame, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    size_mb = out_path.stat().st_size / (1024 ** 2)
    print(f"Saved : {out_path}  ({len(df):,} rows, {size_mb:.2f} MB)")

In [3]:
"""Build the manifest, save the CSV, and show a per-folder distribution."""

df = build_manifest(SOURCE_ROOTS)
save_csv(df, OUT_CSV)

print(f"Total rows   : {len(df):,}")
print(f"Label counts : {df['label'].value_counts().to_dict()}")
print(f"Per dataset  : {df['dataset'].value_counts().to_dict()}")
print()

dist = (
    df.groupby(["dataset", "label"], as_index=False)
      .size()
      .rename(columns={"size": "count"})
      .sort_values(["dataset", "label"])
      .reset_index(drop=True)
)
total_row = pd.DataFrame(
    [{"dataset": "TOTAL", "label": "", "count": int(dist["count"].sum())}]
)
dist = pd.concat([dist, total_row], ignore_index=True)
dist

[scan] /home/taiaburrahman/dataset_manager_pro/data/processed/v9  (dataset=v9)
  v9/fake ->  356,878 images
  v9/real ->  336,374 images
[scan] /home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing  (dataset=v11-processing)
  v11-processing/fake ->  654,414 images
  v11-processing/real ->  142,102 images
Saved : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv/GAID_Dataset_v11_full_train.csv  (1,489,768 rows, 316.97 MB)
Total rows   : 1,489,768
Label counts : {0: 1011292, 1: 478476}
Per dataset  : {'v11-processing': 796516, 'v9': 693252}



,dataset,label,count
0,v11-processing,0,654414
1,v11-processing,1,142102
2,v9,0,356878
3,v9,1,336374
4,TOTAL,,1489768


In [9]:
"""Mini build config.

Reads the full manifest produced above and samples a balanced subset
(equal fake/real per dataset). Then copies the sampled images into
`DEST_ROOT`, preserving the original `<dataset>/<real|fake>/<sub_folder>/` layout.
"""

import shutil
from pathlib import Path

import pandas as pd

FULL_CSV = OUT_DIR / "GAID_Dataset_v11_full_train.csv"
MINI_CSV = OUT_DIR / "GAID_Dataset_v11_mini_train.csv"

TOTAL_ROWS = 10000
RANDOM_SEED = 42

DEST_ROOT = Path("/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11_mini")
DEST_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Full CSV     : {FULL_CSV}")
print(f"Mini CSV     : {MINI_CSV}")
print(f"Target rows  : {TOTAL_ROWS:,}")
print(f"Dest root    : {DEST_ROOT}")


Full CSV     : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv/GAID_Dataset_v11_full_train.csv
Mini CSV     : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv/GAID_Dataset_v11_mini_train.csv
Target rows  : 10,000
Dest root    : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/02.processing/v11_mini


In [10]:
"""Sample a balanced subset: equal fake/real per dataset."""

df_full = pd.read_csv(FULL_CSV)
print(f"Loaded full  : {len(df_full):,} rows")
print(f"Per (dataset,label) available:")
print(df_full.groupby(["dataset", "label"]).size().to_string())
print()

datasets = sorted(df_full["dataset"].unique())
labels = sorted(df_full["label"].unique())
n_groups = len(datasets) * len(labels)
per_group_target = TOTAL_ROWS // n_groups

print(f"Groups       : {n_groups} ({datasets} x {labels})")
print(f"Per group    : {per_group_target:,} (target)")

parts: list[pd.DataFrame] = []
for ds in datasets:
    for lb in labels:
        group = df_full[(df_full["dataset"] == ds) & (df_full["label"] == lb)]
        take = min(per_group_target, len(group))
        if take < per_group_target:
            print(f"  [warn] {ds}/{lb}: only {len(group):,} available (< {per_group_target:,})")
        sampled = group.sample(n=take, random_state=RANDOM_SEED)
        parts.append(sampled)
        print(f"  {ds}/{lb} -> sampled {len(sampled):,}")

df_mini = (
    pd.concat(parts, ignore_index=True)
      .sample(frac=1.0, random_state=RANDOM_SEED)
      .reset_index(drop=True)
)
print()
print(f"Mini rows    : {len(df_mini):,}")
print(f"Label counts : {df_mini['label'].value_counts().to_dict()}")
print(f"Per dataset  : {df_mini['dataset'].value_counts().to_dict()}")

Loaded full  : 1,489,768 rows
Per (dataset,label) available:
dataset         label
v11-processing  0        654414
                1        142102
v9              0        356878
                1        336374

Groups       : 4 (['v11-processing', 'v9'] x [np.int64(0), np.int64(1)])
Per group    : 2,500 (target)
  v11-processing/0 -> sampled 2,500
  v11-processing/1 -> sampled 2,500
  v9/0 -> sampled 2,500
  v9/1 -> sampled 2,500

Mini rows    : 10,000
Label counts : {0: 5000, 1: 5000}
Per dataset  : {'v9': 5000, 'v11-processing': 5000}


In [11]:
"""Copy sampled images to DEST_ROOT preserving `<dataset>/<real|fake>/...` layout,
then save the mini CSV with paths rewritten to the new location.

The relative destination path is taken from the source path itself, anchored
at the `dataset` segment (e.g. `.../processed/v9/fake/sub/foo.jpg` with
dataset=`v9` becomes `<DEST_ROOT>/v9/fake/sub/foo.jpg`).
"""

from tqdm.auto import tqdm


def relative_from_dataset(src: Path, dataset: str) -> Path:
    parts = src.parts
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == dataset:
            return Path(*parts[i:])
    raise ValueError(f"dataset segment '{dataset}' not found in {src}")


new_paths: list[str] = []
n_copied = 0
n_skipped = 0
n_missing = 0

for src_str, dataset in tqdm(
    zip(df_mini["filepath"].tolist(), df_mini["dataset"].tolist()),
    total=len(df_mini),
    desc="copy",
):
    src = Path(src_str)
    if not src.exists():
        n_missing += 1
        new_paths.append("")
        continue

    rel = relative_from_dataset(src, dataset)  # e.g. v9/fake/sub/foo.jpg
    dst = DEST_ROOT / rel
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        n_skipped += 1
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        n_copied += 1
    new_paths.append(str(dst))

df_mini_out = df_mini.copy()
df_mini_out["filepath"] = new_paths

before = len(df_mini_out)
df_mini_out = df_mini_out[df_mini_out["filepath"] != ""].reset_index(drop=True)
dropped = before - len(df_mini_out)

save_csv(df_mini_out, MINI_CSV)
print(f"Copied       : {n_copied:,}")
print(f"Skipped      : {n_skipped:,} (already present)")
print(f"Missing src  : {n_missing:,} (rows dropped: {dropped:,})")
print()

dist_mini = (
    df_mini_out.groupby(["dataset", "label"], as_index=False)
      .size()
      .rename(columns={"size": "count"})
      .sort_values(["dataset", "label"])
      .reset_index(drop=True)
)
total_row = pd.DataFrame(
    [{"dataset": "TOTAL", "label": "", "count": int(dist_mini["count"].sum())}]
)
dist_mini = pd.concat([dist_mini, total_row], ignore_index=True)
dist_mini

copy:   0%|          | 0/10000 [00:00<?, ?it/s]

Saved : /home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv/GAID_Dataset_v11_mini_train.csv  (10,000 rows, 2.31 MB)
Copied       : 10,000
Skipped      : 0 (already present)
Missing src  : 0 (rows dropped: 0)



,dataset,label,count
0,v11-processing,0,2500
1,v11-processing,1,2500
2,v9,0,2500
3,v9,1,2500
4,TOTAL,,10000
